# Build feature-vector snapshots from trades

`fiml` accepts an already-loaded, globally ordered trade DataFrame. File loading stays in pandas.

In [ ]:
import numpy as np
import pandas as pd
import fiml

In [ ]:
trades = pd.read_csv("trades.csv")
trades

In [ ]:
feature_set = (fiml.FeatureSet()
    .obv_timed("BTCUSDT", aggregation="1ms", window="60s")
    .trade_count_timed("BTCUSDT", aggregation="1ms", window="60s")
    .sma("BTCUSDT", period=2, event_kind="trade")
    .ema("BTCUSDT", period=2, event_kind="trade")
    .day_of_week("BTCUSDT"))
extractor = fiml.FeatureExtractor(feature_set, output_dtype=np.float32)

In [ ]:
features = extractor.compute_features(trades)
features

In [ ]:
assert features.index.equals(trades.index)
assert features["symbol"].equals(trades["symbol"])
assert features["ts"].equals(trades["ts"])
assert list(features.columns[2:]) == extractor.feature_names()
assert all(dtype == np.float32 for dtype in features.dtypes.iloc[2:])
assert features.shape == (len(trades), extractor.n_features() + 2)
assert not features[["sma_2", "ema_2"]].isna().any().any()